In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# The Neural Network
class QuantumOscillatorNN(nn.Module):
    def __init__(self):
        super(QuantumOscillatorNN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 128), nn.Tanh(),
            nn.Linear(128, 128), nn.Tanh(),
            nn.Linear(128, 128), nn.Tanh(),
            nn.Linear(128, 2)
        )

    def forward(self, x, t):
        return self.net(torch.cat([x, t], dim=1))

In [ ]:
# Physics Equations & Loss
def get_potential(x):
    return 0.5 * (x**2)

def compute_loss(model, x_ic, t_ic, u_ic, v_ic, x_c, t_c, x_b, t_b):
    # Initial Condition Loss 
    psi_ic = model(x_ic, t_ic)
    loss_ic = torch.mean((psi_ic[:, 0:1] - u_ic)**2) + torch.mean((psi_ic[:, 1:2] - v_ic)**2)

    # Boundary Condition Loss
    psi_bc = model(x_b, t_b)
    loss_bc = torch.mean(psi_bc**2)

    # Schrödinger PDE Loss
    x_c.requires_grad = True
    t_c.requires_grad = True
    psi = model(x_c, t_c)
    u, v = psi[:, 0:1], psi[:, 1:2]
    V = get_potential(x_c)

    u_t = torch.autograd.grad(u, t_c, torch.ones_like(u), create_graph=True)[0]
    v_t = torch.autograd.grad(v, t_c, torch.ones_like(v), create_graph=True)[0]
    u_x = torch.autograd.grad(u, x_c, torch.ones_like(u), create_graph=True)[0]
    v_x = torch.autograd.grad(v, x_c, torch.ones_like(v), create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x, x_c, torch.ones_like(u_x), create_graph=True)[0]
    v_xx = torch.autograd.grad(v_x, x_c, torch.ones_like(v_x), create_graph=True)[0]

    f_u = u_t + 0.5 * v_xx - V * v
    f_v = v_t - 0.5 * u_xx + V * u
    loss_pde = torch.mean(f_u**2) + torch.mean(f_v**2)

    # Normalization Loss
    # We check 10 random time slices
    t_norm = torch.linspace(0, 2*np.pi, 10).to(x_ic.device)
    x_norm = torch.linspace(-5, 5, 100).to(x_ic.device)
    X_n, T_n = torch.meshgrid(x_norm, t_norm, indexing='ij')
    psi_norm = model(X_n.reshape(-1,1), T_n.reshape(-1,1))
    prob_dist = psi_norm[:,0]**2 + psi_norm[:,1]**2
    # Integration approximation: mean * domain_width
    total_prob = torch.mean(prob_dist.reshape(100, 10)) * 10 
    loss_norm = torch.mean((total_prob - 1.0)**2)

    # Balanced Total Loss
    return 25*loss_ic + 15*loss_pde + 5*loss_bc + 60*loss_norm



In [ ]:
# Data Generation
def generate_data():
    x_ic = torch.linspace(-5, 5, 400).view(-1, 1)
    t_ic = torch.zeros_like(x_ic)
    x0 = -2.0
    u_ic = (np.pi**(-0.25)) * torch.exp(-0.5 * (x_ic - x0)**2)
    v_ic = torch.zeros_like(u_ic)

    # Collocation points 
    x_c = torch.distributions.Uniform(-5, 5).sample((10000, 1))
    t_c = torch.distributions.Uniform(0, 2*np.pi).sample((10000, 1))

    # Boundaries: Randomly pick -5.0 or 5.0 for 500 points
    t_b = torch.distributions.Uniform(0, 2*np.pi).sample((500, 1))
    boundary_values = torch.tensor([-5.0, 5.0])
    indices = torch.randint(0, 2, (500, 1))
    x_b = boundary_values[indices]
    
    return x_ic, t_ic, u_ic, v_ic, x_c, t_c, x_b, t_b


In [ ]:
# Training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = QuantumOscillatorNN().to(device)

# Prepare Data
x_ic, t_ic, u_ic, v_ic, x_c, t_c, x_b, t_b = [v.to(device) for v in generate_data()]

# Adam Optimizer
optimizer_adam = torch.optim.Adam(model.parameters(), lr=1e-3)
print("Stage 1: Adam Training...")
for epoch in range(5001):
    optimizer_adam.zero_grad()
    loss = compute_loss(model, x_ic, t_ic, u_ic, v_ic, x_c, t_c, x_b, t_b)
    loss.backward()
    optimizer_adam.step()
    if epoch % 1000 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.6f}")

# L-BFGS Optimizer 
optimizer_lbfgs = torch.optim.LBFGS(model.parameters(), max_iter=20000, tolerance_grad=1e-7, history_size=50)
print("\nStage 2: L-BFGS Training (High Precision)...")

def closure():
    optimizer_lbfgs.zero_grad()
    loss = compute_loss(model, x_ic, t_ic, u_ic, v_ic, x_c, t_c, x_b, t_b)
    loss.backward()
    return loss

optimizer_lbfgs.step(closure)
print("Training Complete.")

In [ ]:
# Visualization of the Results
def plot_results():
    x = np.linspace(-5, 5, 200)
    t_vals = [0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi]
    plt.figure(figsize=(16, 4))
    
    for i, t_v in enumerate(t_vals):
        tx = torch.ones(200, 1).to(device) * t_v
        xx = torch.tensor(x, dtype=torch.float32).view(-1, 1).to(device)
        with torch.no_grad():
            res = model(xx, tx)
            prob = (res[:,0]**2 + res[:,1]**2).cpu().numpy()
        
        plt.subplot(1, 5, i+1)
        plt.plot(x, prob, color='darkblue', lw=2)
        plt.fill_between(x, prob, alpha=0.3, color='blue')
        plt.plot(x, 0.1*x**2, '--', color='red', alpha=0.3)
        plt.title(f"t = {i}/4 Period")
        plt.ylim(0, 1)
        plt.xlim(-5, 5)
        plt.savefig("wave_evolution.png",dpi=600)
    plt.show()

plot_results()

In [ ]:
#Verification
def verify_physics_conservation(model):
    print("\n--- Physics Verification: Total Probability ---")
    
    x_fine = torch.linspace(-5, 5, 1000).view(-1, 1).to(device)
    dx = x_fine[1] - x_fine[0] # The width of each small slice
    test_times = [0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi]
    time_labels = ["0 (Start)", "1/4 Period", "1/2 Period", "3/4 Period", "Full Cycle"]
    
    for t_val, label in zip(test_times, time_labels):
        t_vec = torch.ones_like(x_fine) * t_val
        
        with torch.no_grad():
            psi = model(x_fine, t_vec)
            # Probability Density
            prob_density = psi[:, 0]**2 + psi[:, 1]**2
            
            # Numerical Integration (Riemann Sum)
            total_probability = torch.sum(prob_density) * dx
            
        print(f"Time: {label:12} | Sum of Probabilities: {total_probability.item():.6f}")

verify_physics_conservation(model)